# Mapas do artigo do açaí (Figuras 2, 3 e 4) — versão para a revista

**Antes de rodar:** a chave do Copernicus deve estar nos *Secrets* do Colab (ícone de chave 🔑 na barra lateral) com o nome `CDS_KEY` e com o acesso ao notebook ligado.

Depois é só **Ambiente de execução → Executar tudo**. As figuras ficam em `MyDrive/artigo_acai/mapas/`.

O que mudou em relação ao notebook antigo:
- chave lida dos *Secrets* (não aparece no código);
- rótulos em inglês, escala, norte, datum SIRGAS 2000 e fonte da base cartográfica (IBGE, malha municipal 2022);
- a mesma escala de cores nas 4 estações (antes cada painel tinha a sua, o que impedia comparar);
- sem interpolar valores sobre a água (o ERA5-Land não tem dado ali);
- DJF = dezembro de 2023 a fevereiro de 2024, igual à Tabela 1;
- precipitação = total diário (antes o título dizia 15 h por engano); temperatura = 15 h local.


In [ ]:
!pip install -q cdsapi netcdf4 h5netcdf cartopy geobr geopandas

In [ ]:
import os, calendar, warnings
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from matplotlib import colors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cdsapi
warnings.filterwarnings('ignore')
plt.rcParams.update({'font.family': 'DejaVu Serif', 'font.size': 11})

from google.colab import userdata, drive
CDS_KEY = userdata.get('CDS_KEY')          # chave nova, guardada nos Secrets
client = cdsapi.Client(url='https://cds.climate.copernicus.eu/api', key=CDS_KEY)

drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/artigo_acai/mapas'
os.makedirs(OUT, exist_ok=True)

In [ ]:
# Domínio do estudo (o mesmo da normal climatológica e da Tabela 1)
N, W, S, E = -1.4, -49.8, -2.6, -48.5
AREA = [N, W, S, E]
EXT = [W - 0.05, E + 0.07, S - 0.05, N + 0.05]   # inclui as bordas das células de 0,1°

TOWNS = {'Belém': (-48.4902, -1.4558), 'Abaetetuba': (-48.8788, -1.7217),
         'Igarapé-Miri': (-48.9578, -1.9987), 'Cametá': (-49.4956, -2.2504)}

SEASONS = {   # (ano, mês)
    'DJF (Dec 2023–Feb 2024)': [(2023, 12), (2024, 1), (2024, 2)],
    'MAM 2024': [(2024, 3), (2024, 4), (2024, 5)],
    'JJA 2024': [(2024, 6), (2024, 7), (2024, 8)],
    'SON 2024': [(2024, 9), (2024, 10), (2024, 11)],
}
CRS = ccrs.PlateCarree()
BASE_NOTE = 'Geographic coordinates, datum SIRGAS 2000. Base: IBGE municipal mesh (2022); climate: ERA5-Land.'

In [ ]:
# Download: médias mensais de 2023 e 2024 (arquivos pequenos, poucos minutos)
def get(product, variables, time, fname):
    path = f'/content/{fname}'
    if not os.path.exists(path):
        client.retrieve('reanalysis-era5-land-monthly-means', {
            'product_type': [product], 'variable': variables,
            'year': ['2023', '2024'], 'month': [f'{m:02d}' for m in range(1, 13)],
            'time': [time], 'area': AREA,
            'data_format': 'netcdf', 'download_format': 'unarchived'}).download(path)
    ds = xr.open_dataset(path)
    t = 'valid_time' if 'valid_time' in ds.dims else 'time'
    return ds.rename({t: 'time'})

ds_p = get('monthly_averaged_reanalysis', ['total_precipitation'], '00:00', 'era5land_precip_mensal.nc')
ds_t = get('monthly_averaged_reanalysis_by_hour_of_day', ['2m_temperature'], '18:00', 'era5land_t2m_18utc.nc')
print(ds_p, ds_t)

In [ ]:
# Médias sazonais (precipitação em mm/dia; temperatura das 15 h locais em °C)
def seasonal(da):
    out = {}
    for name, ym in SEASONS.items():
        sel = [pd.Timestamp(y, m, 1) for y, m in ym]
        tt = pd.to_datetime(da.time.values)
        idx = [i for i, x in enumerate(tt) if pd.Timestamp(x.year, x.month, 1) in sel]
        assert len(idx) == 3, f'faltam meses em {name}'
        out[name] = da.isel(time=idx).mean('time')
    return out

PREC = seasonal(ds_p['tp'] * 1000)
TEMP = seasonal(ds_t['t2m'] - 273.15)

# Conferência com a Tabela 1 do artigo (médias do domínio)
print(pd.DataFrame({'Precip (mm/dia)': {k: float(v.mean()) for k, v in PREC.items()},
                    'T 15h (°C)': {k: float(v.mean()) for k, v in TEMP.items()}}).round(2))

In [ ]:
# Base cartográfica do IBGE (malha municipal 2022, SIRGAS 2000) via geobr
try:
    import geobr
    MUN = geobr.read_municipality(code_muni='PA', year=2022)
    UF = geobr.read_state(year=2020)
    print('Malha do IBGE carregada:', len(MUN), 'municípios do Pará')
except Exception as e:
    MUN = UF = None
    BASE_NOTE = BASE_NOTE.replace('IBGE municipal mesh (2022)', 'Natural Earth')
    print('geobr indisponível, usando Natural Earth:', e)

In [ ]:
# Elementos de mapa: escala, norte, municípios, grade
def scale_bar(ax, km=25, loc=(0.05, 0.06)):
    x0, x1, y0, y1 = ax.get_extent(CRS)
    lat = y0 + loc[1] * (y1 - y0)
    lon0 = x0 + loc[0] * (x1 - x0)
    dlon = km / (111.32 * np.cos(np.radians(lat)))
    for i, c in enumerate(['k', 'w']):
        ax.add_patch(plt.Rectangle((lon0 + i * dlon / 2, lat), dlon / 2, (y1 - y0) * 0.012,
                     facecolor=c, edgecolor='k', lw=0.8, transform=CRS, zorder=10))
    for frac, lab in [(0, '0'), (1, f'{km} km')]:
        ax.text(lon0 + frac * dlon, lat + (y1 - y0) * 0.025, lab, ha='center', va='bottom',
                fontsize=9, transform=CRS, zorder=10,
                bbox=dict(fc='white', ec='none', alpha=0.7, pad=0.5))

def north_arrow(ax, xy=(0.92, 0.86)):
    ax.annotate('N', xy=xy, xytext=(xy[0], xy[1] - 0.12), xycoords='axes fraction',
                ha='center', va='center', fontsize=12, fontweight='bold',
                arrowprops=dict(facecolor='k', width=4, headwidth=11, headlength=10), zorder=11)

def base_layers(ax, labels=True, fs=9):
    if MUN is not None:
        MUN.boundary.plot(ax=ax, color='0.25', lw=0.5, transform=CRS, zorder=5)
    else:
        ax.add_feature(cfeature.COASTLINE.with_scale('10m'), lw=0.6, zorder=5)
    for name, (lon, lat) in TOWNS.items():
        ax.plot(lon, lat, marker='^', ms=7, mfc='k', mec='white', transform=CRS, zorder=8)
        if labels:
            ax.text(lon + 0.025, lat + 0.02, name, fontsize=fs, fontweight='bold', transform=CRS, zorder=9,
                    bbox=dict(fc='white', ec='none', alpha=0.75, pad=0.6))

def grid(ax, left=True, bottom=True, step=0.4):
    gl = ax.gridlines(crs=CRS, draw_labels=True, lw=0.4, color='0.5', alpha=0.5, ls='--',
                      xlocs=np.arange(-50, -48, step), ylocs=np.arange(-3, -1, step))
    gl.top_labels = gl.right_labels = False
    gl.left_labels, gl.bottom_labels = left, bottom
    gl.xlabel_style = gl.ylabel_style = {'size': 9}

In [ ]:
# FIGURA 2 — localização
fig = plt.figure(figsize=(12, 6.2))
ax1 = fig.add_subplot(1, 2, 1, projection=CRS)
ax1.set_extent([-59.5, -45.5, -10.5, 3.0], CRS)
ax1.add_feature(cfeature.OCEAN, facecolor='#d9ecf5')
if UF is not None:
    UF.plot(ax=ax1, facecolor='#f2efe6', edgecolor='0.55', lw=0.5, transform=CRS)
    UF[UF.abbrev_state == 'PA'].plot(ax=ax1, facecolor='#e3d9b8', edgecolor='k', lw=1.0, transform=CRS)
else:
    ax1.add_feature(cfeature.LAND, facecolor='#f2efe6')
    ax1.add_feature(cfeature.STATES, edgecolor='0.55', lw=0.5)
ax1.add_feature(cfeature.RIVERS.with_scale('50m'), edgecolor='#4a90b8', lw=0.6)
ax1.plot([W, E, E, W, W], [S, S, N, N, S], color='red', lw=2, transform=CRS, label='Study domain')
ax1.plot(-48.4902, -1.4558, marker='*', ms=12, mfc='k', mec='white', transform=CRS, label='Belém (state capital)')
ax1.text(-53.5, -5.0, 'PARÁ', fontsize=13, fontweight='bold', ha='center', transform=CRS, color='0.25')
ax1.legend(loc='lower left', fontsize=9, framealpha=0.9)
gl = ax1.gridlines(crs=CRS, draw_labels=True, lw=0.4, ls='--', color='0.5', alpha=0.5,
                   xlocs=range(-58, -44, 4), ylocs=range(-10, 4, 4))
gl.top_labels = gl.right_labels = False
ax1.set_title('(a) Pará State, Brazil', loc='left', fontsize=12)
scale_bar(ax1, km=300); north_arrow(ax1)

# mapa de localização do Brasil (inset)
ins = fig.add_axes([0.355, 0.62, 0.13, 0.25], projection=CRS)
ins.set_extent([-75, -33, -34, 6], CRS)
ins.add_feature(cfeature.OCEAN, facecolor='#d9ecf5')
if UF is not None:
    UF.plot(ax=ins, facecolor='#f2efe6', edgecolor='0.6', lw=0.2, transform=CRS)
    UF[UF.abbrev_state == 'PA'].plot(ax=ins, facecolor='#c8b77a', edgecolor='k', lw=0.4, transform=CRS)
else:
    ins.add_feature(cfeature.LAND, facecolor='#f2efe6'); ins.add_feature(cfeature.BORDERS, lw=0.3)
ins.text(-52, -14, 'BRAZIL', fontsize=7, ha='center', transform=CRS)

ax2 = fig.add_subplot(1, 2, 2, projection=CRS)
ax2.set_extent(EXT, CRS)
ax2.set_facecolor('#d9ecf5')
if MUN is not None:
    MUN.plot(ax=ax2, facecolor='#f2efe6', edgecolor='none', transform=CRS)
else:
    ax2.add_feature(cfeature.LAND.with_scale('10m'), facecolor='#f2efe6')
base_layers(ax2, fs=10)
ax2.plot([W, E, E, W, W], [S, S, N, N, S], color='red', lw=2, transform=CRS)
ax2.set_title('(b) Lower Tocantins study domain', loc='left', fontsize=12)
grid(ax2); scale_bar(ax2, km=25); north_arrow(ax2)
fig.text(0.5, 0.01, BASE_NOTE, ha='center', fontsize=9)
fig.savefig(f'{OUT}/Figure2_study_area.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# FIGURAS 3 e 4 — campos sazonais com a MESMA escala nos 4 painéis
def season_maps(fields, cmap, label, fname, step):
    allv = np.concatenate([v.values.ravel() for v in fields.values()])
    allv = allv[np.isfinite(allv)]
    lo, hi = np.floor(allv.min() / step) * step, np.ceil(allv.max() / step) * step
    levels = np.arange(lo, hi + step / 2, step)
    norm = colors.BoundaryNorm(levels, plt.get_cmap(cmap).N)
    fig, axs = plt.subplots(2, 2, figsize=(11, 10.2), subplot_kw={'projection': CRS})
    for i, (ax, (name, da)) in enumerate(zip(axs.flat, fields.items())):
        ax.set_extent(EXT, CRS)
        ax.set_facecolor('#e6e6e6')      # água / sem dado
        pm = ax.pcolormesh(da.longitude, da.latitude, da, cmap=cmap, norm=norm,
                           shading='nearest', transform=CRS, zorder=2)
        base_layers(ax, labels=True, fs=8)
        ax.plot([W, E, E, W, W], [S, S, N, N, S], color='k', lw=0.8, transform=CRS, zorder=6)
        ax.set_title(f'({"abcd"[i]}) {name}', loc='left', fontsize=12)
        grid(ax, left=(i % 2 == 0), bottom=(i >= 2))
        if i == 0:
            scale_bar(ax, km=25); north_arrow(ax)
    fig.subplots_adjust(left=0.07, right=0.97, top=0.96, bottom=0.14, wspace=0.08, hspace=0.12)
    cax = fig.add_axes([0.2, 0.075, 0.6, 0.022])
    cb = fig.colorbar(pm, cax=cax, orientation='horizontal', ticks=levels[::2] if len(levels) > 12 else levels)
    cb.set_label(label, fontsize=12)
    fig.text(0.5, 0.015, BASE_NOTE + ' Grey: water (no ERA5-Land data).', ha='center', fontsize=9)
    fig.savefig(f'{OUT}/{fname}.png', dpi=300)
    plt.show()

season_maps(PREC, 'Blues', 'Precipitation (mm day$^{-1}$)', 'Figure3_precipitation', step=1.0)
season_maps(TEMP, 'YlOrRd', 'Air temperature at 15:00 local time (°C)', 'Figure4_temperature_15h', step=0.5)
print('Figuras salvas em', OUT)